# 🧩 Mini-Lab: Seam Teardown

**Module: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** where vendor-specific API calls are tangled with business logic
2. **Explain** why tightly coupled vendor calls make code fragile
3. **Recognize** where to place an interface seam to isolate the AI dependency

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. |


## The Scenario

Imagine you inherited a small codebase that classifies customer support tickets. The original developer hard-coded the OpenAI API call **directly inside** the business logic. One day the vendor changes its API, raises prices, or goes down — and the entire application breaks.

We'll look at this "bad" code, pinpoint exactly where the vendor dependency leaks in, and then mark where the **seam** should go.

## Step 1 — The Tightly-Coupled Code

Below is a simplified ticket classifier. Read through it and notice how many lines **know about the vendor** (OpenAI-specific imports, model names, response shapes).

In [1]:
# ❌ Tightly-coupled code — the vendor is everywhere

COUPLED_CODE = '''
import openai                                       # 🔴 vendor import

def classify_ticket(ticket_text: str) -> str:
    """Classify a support ticket into a category."""

    # --- business logic: build the prompt ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\\n"
        f"Ticket: {ticket_text}\\n"
        f"Category:"
    )

    # --- vendor call buried in the function ---
    client = openai.OpenAI()                        # 🔴 vendor client
    response = client.chat.completions.create(      # 🔴 vendor API shape
        model="gpt-4o-mini",                        # 🔴 vendor model name
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content       # 🔴 vendor response shape

    # --- business logic: validate the answer ---
    category = raw.strip().lower()
    if category not in categories:
        category = "other"
    return category
'''

print(COUPLED_CODE)


import openai                                       # 🔴 vendor import

def classify_ticket(ticket_text: str) -> str:
    """Classify a support ticket into a category."""

    # --- business logic: build the prompt ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\n"
        f"Ticket: {ticket_text}\n"
        f"Category:"
    )

    # --- vendor call buried in the function ---
    client = openai.OpenAI()                        # 🔴 vendor client
    response = client.chat.completions.create(      # 🔴 vendor API shape
        model="gpt-4o-mini",                        # 🔴 vendor model name
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content       # 🔴 vendor response shape

    # --- business logic: validate the answer ---
    category = raw.strip().lower()
    if category not in categories:
        category = "oth

## Step 2 — Counting the Vendor Touch-Points

Let's systematically find every line that is **vendor-specific**. If OpenAI disappeared tomorrow, which lines would break?

In [2]:
# Lines that reference the vendor (OpenAI)
vendor_lines = [
    ("import openai",                        "Vendor import — ties file to openai package"),
    ("client = openai.OpenAI()",             "Vendor client instantiation"),
    ("client.chat.completions.create(...)",  "Vendor-specific API method"),
    ('model="gpt-4o-mini"',                  "Vendor-specific model name"),
    ("response.choices[0].message.content",  "Vendor-specific response shape"),
]

print(f"Vendor touch-points found: {len(vendor_lines)}\n")
for line, reason in vendor_lines:
    print(f"  🔴  {line}")
    print(f"      └─ {reason}\n")

Vendor touch-points found: 5

  🔴  import openai
      └─ Vendor import — ties file to openai package

  🔴  client = openai.OpenAI()
      └─ Vendor client instantiation

  🔴  client.chat.completions.create(...)
      └─ Vendor-specific API method

  🔴  model="gpt-4o-mini"
      └─ Vendor-specific model name

  🔴  response.choices[0].message.content
      └─ Vendor-specific response shape



Five vendor-specific lines are **mixed right in** with the business logic (building the prompt, validating the category). That's the problem: there is no boundary — no **seam** — between *what the app does* and *which AI service it calls*.

## Step 3 — What Breaks When the Vendor Changes?

Let's make this concrete. Suppose the vendor changes its response format (a real thing that happens with API version updates).

In [3]:
# Simulating a vendor API change

# OLD response shape (v1)
old_response = {
    "choices": [{"message": {"content": "billing"}}]
}

# NEW response shape (v2) — vendor renamed the field
new_response = {
    "output": [{"text": "billing"}]
}

# The tightly-coupled code expects the OLD shape
try:
    result = old_response["choices"][0]["message"]["content"]
    print(f"✅ Old format works: '{result}'")
except (KeyError, IndexError) as e:
    print(f"❌ Old format broke: {e}")

try:
    result = new_response["choices"][0]["message"]["content"]
    print(f"✅ New format works: '{result}'")
except (KeyError, IndexError) as e:
    print(f"❌ New format broke: {e}")

print("\n→ The vendor change broke classify_ticket() even though")
print("  the business logic (categories, validation) didn't change.")

✅ Old format works: 'billing'
❌ New format broke: 'choices'

→ The vendor change broke classify_ticket() even though
  the business logic (categories, validation) didn't change.


## Step 4 — Drawing the Seam

An **interface seam** is the line you draw between "business logic" and "vendor call." Everything above the seam is *your application*. Everything below the seam is *the replaceable AI backend*.

Let's label each part of the original function.

In [4]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

# Labeling each section of classify_ticket()

business_lines = [
    "categories = ['billing', 'technical', 'account', 'other']",
    "prompt = f'Classify this support ticket into one of {categories}...'",
    "# (prompt construction is business logic — it knows WHAT to ask)",
    "category = raw.strip().lower()",
    "if category not in categories: category = 'other'",
]

vendor_only_lines = [
    "client = openai.OpenAI()",
    "response = client.chat.completions.create(model='gpt-4o-mini', ...)",
    "raw = response.choices[0].message.content",
]

diagram = """
## Function Anatomy with Seam

```
def classify_ticket(ticket_text):
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • define categories                 │
    │  │  • build the prompt                  │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  VENDOR CODE (replaceable)           │
    │  │  • create OpenAI client              │
    │  │  • call chat.completions.create()    │
    │  │  • dig into response.choices[0]...   │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • validate / clean the answer       │
    │  │  • return the category               │
    │  └─────────────────────────────────────┘
```

The vendor block in the middle is the part you extract behind an interface.
"""

md(diagram)


## Function Anatomy with Seam

```
def classify_ticket(ticket_text):
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • define categories                 │
    │  │  • build the prompt                  │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  VENDOR CODE (replaceable)           │
    │  │  • create OpenAI client              │
    │  │  • call chat.completions.create()    │
    │  │  • dig into response.choices[0]...   │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • validate / clean the answer       │
    │  │  • return the category               │
    │  └─────────────────────────────────────┘
```

The vendor block in the middle is the part you extract behind an interface.


## Step 5 — What the Seam Enables

Once you extract the vendor block behind an interface, the business logic only knows:

> *"I give you a prompt string, you give me back a response string."*

It doesn't care **who** answers — OpenAI, Anthropic, a local Ollama model, or a mock that returns `"billing"` every time for testing.

In [5]:
# What the business logic SHOULD look like after placing the seam

after_seam = """
## After Placing the Seam

```python
def classify_ticket(ticket_text: str, ai_client) -> str:
    # --- business logic only ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\\n"
        f"Ticket: {ticket_text}\\n"
        f"Category:"
    )

    # ONE call through the seam — no vendor details here
    raw = ai_client.complete(prompt)   # ← the interface seam

    # --- business logic continues ---
    category = raw.strip().lower()
    if category not in categories:
        category = "other"
    return category
```

**Zero vendor imports. Zero vendor response parsing.** The function works
with *any* backend that has a `.complete(prompt) -> str` method.
"""

md(after_seam)


## After Placing the Seam

```python
def classify_ticket(ticket_text: str, ai_client) -> str:
    # --- business logic only ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\n"
        f"Ticket: {ticket_text}\n"
        f"Category:"
    )

    # ONE call through the seam — no vendor details here
    raw = ai_client.complete(prompt)   # ← the interface seam

    # --- business logic continues ---
    category = raw.strip().lower()
    if category not in categories:
        category = "other"
    return category
```

**Zero vendor imports. Zero vendor response parsing.** The function works
with *any* backend that has a `.complete(prompt) -> str` method.


## Step 6 — Quick Checklist: Spotting a Missing Seam

Use this checklist on any codebase to detect a missing interface seam.

In [6]:
checklist = [
    ("import openai / import anthropic in business-logic files",
     "Vendor import should only appear in the backend module"),
    ("Model name hard-coded in the function",
     "Model selection belongs in configuration, not logic"),
    ("response.choices[0].message.content deep in your code",
     "Response parsing belongs inside the backend adapter"),
    ("Cannot run tests without a real API key",
     "A seam lets you inject a mock — no key needed"),
    ("Changing provider requires editing many files",
     "With a seam, you only change the backend module"),
]

print("🔍 Missing-Seam Checklist\n")
for i, (symptom, fix) in enumerate(checklist, 1):
    print(f"  {i}. Symptom:  {symptom}")
    print(f"     Fix:      {fix}\n")

🔍 Missing-Seam Checklist

  1. Symptom:  import openai / import anthropic in business-logic files
     Fix:      Vendor import should only appear in the backend module

  2. Symptom:  Model name hard-coded in the function
     Fix:      Model selection belongs in configuration, not logic

  3. Symptom:  response.choices[0].message.content deep in your code
     Fix:      Response parsing belongs inside the backend adapter

  4. Symptom:  Cannot run tests without a real API key
     Fix:      A seam lets you inject a mock — no key needed

  5. Symptom:  Changing provider requires editing many files
     Fix:      With a seam, you only change the backend module



## 🎯 Summary

### Key Takeaways

1. **Interface seam** — the boundary between your business logic and the vendor-specific AI call, placed so neither side knows about the other's internals
2. **Vendor lock-in is fragile** — hard-coding API imports, model names, and response shapes throughout your code means any vendor change breaks everything
3. **Spotting the seam** — look for vendor imports, hard-coded model names, and vendor-specific response parsing mixed into business logic
4. **One-method boundary** — after placing the seam, the business logic only needs a simple `complete(prompt) → str` call, making the AI backend fully replaceable